In [2]:
import requests
import getpass
import pandas as pd
from io import StringIO
import pyspark
from pyspark.sql.functions import col
from pyspark.sql.functions import lit
from pyspark.sql import functions as F

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 5, Finished, Available, Finished, False)

In [3]:
path = "Files/bronze/reference/taxi_zone_lookup.csv"
df =(spark.read.option("header", True)).csv(path)
df.show()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 6, Finished, Available, Finished, False)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [4]:
print(df.dtypes)
print(df.filter(col("LocationID").isNull()).count())
print(df.select("LocationID").distinct().count())
print(df.select("LocationID").count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 7, Finished, Available, Finished, False)

[('LocationID', 'string'), ('Borough', 'string'), ('Zone', 'string'), ('service_zone', 'string')]
0
265
265


In [5]:
#changing LOcation ID to int
df = df.withColumn("LocationID", col("LocationID").cast("int"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 8, Finished, Available, Finished, False)

In [6]:
df.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_taxi_zones")

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 9, Finished, Available, Finished, False)

In [7]:
#Checking and fixing taxi data
taxi_data_examples = {}

for year in [2023,2024,2025]:
    path = f"Files/bronze/nyc_taxi/{year}/01/yellow_tripdata_{year}-01.parquet"

    taxi_data_examples[year] = spark.read.parquet(path)
    print(year)
    taxi_data_examples[year].printSchema()



StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 10, Finished, Available, Finished, False)

2023
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

2024
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropo

In [8]:
display(taxi_data_examples[2025].limit(10))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f251f195-7cc0-4131-a42c-04ea49078fe4)

In [9]:
#loading all taxi data
taxi_data = {}

for year in [2023,2024,2025]:
    for month in range(1, 13):
        path = f"Files/bronze/nyc_taxi/{year}/{month:02d}/yellow_tripdata_{year}-{month:02d}.parquet"
        taxi_data[f"{year}-{month:02d}"] = spark.read.parquet(path)

print(len(taxi_data))




StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 12, Finished, Available, Finished, False)

36


In [10]:
for month in taxi_data:
    part = taxi_data[month]
    for column in part.columns:
        part = part.withColumnRenamed(column, column.lower())

taxi_data[month] = part

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 13, Finished, Available, Finished, False)

In [11]:
print(taxi_data["2025-01"].columns)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 14, Finished, Available, Finished, False)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


In [12]:
print(taxi_data["2023-01"].columns)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 15, Finished, Available, Finished, False)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [13]:
for month in taxi_data:
    cells_data = taxi_data[month]

    cells_data = cells_data.withColumn("vendorid", col("vendorid").cast("int"))
    cells_data = cells_data.withColumn("pulocationid", col("pulocationid").cast("int"))
    cells_data = cells_data.withColumn("dolocationid", col("dolocationid").cast("int"))
    cells_data = cells_data.withColumn("passenger_count", col("passenger_count").cast("double"))
    cells_data = cells_data.withColumn("ratecodeid", col("ratecodeid").cast("double"))

    taxi_data[month] = cells_data

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 16, Finished, Available, Finished, False)

In [14]:
print(taxi_data["2023-01"].printSchema())
print(taxi_data["2024-01"].printSchema())
print(taxi_data["2025-01"].printSchema())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 17, Finished, Available, Finished, False)

root
 |-- vendorid: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

None
root
 |-- vendorid: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_d

In [15]:
#uniting tables
months = list(taxi_data)

df_taxi = taxi_data[months[0]]

for month in months[1:]:
    df_taxi = df_taxi.unionByName(
        taxi_data[month],
        allowMissingColumns=True
    )

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 18, Finished, Available, Finished, False)

In [16]:
#checking united data
print(df_taxi.count())
df_taxi.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "total_amount").summary("count", "min","max").show()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 19, Finished, Available, Finished, False)

128202548
+-------+-------------+------------+
|summary|trip_distance|total_amount|
+-------+-------------+------------+
|  count|    128202548|   128202548|
|    min|          0.0|    -2265.45|
|    max|    398608.62|   863380.37|
+-------+-------------+------------+



In [17]:
df_taxi.agg(
    F.min("tpep_pickup_datetime").alias("first_pickup"),
    F.max("tpep_pickup_datetime").alias("last_pickup"),
    F.min("tpep_dropoff_datetime").alias("first_dropoff"),
    F.max("tpep_dropoff_datetime").alias("last_dropoff"),
    F.count("tpep_pickup_datetime").alias("filled_pickup"),
    F.count("tpep_dropoff_datetime").alias("filled_dropoff")
).show(truncate=False)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 20, Finished, Available, Finished, False)

+-------------------+-------------------+-------------------+-------------------+-------------+--------------+
|first_pickup       |last_pickup        |first_dropoff      |last_dropoff       |filled_pickup|filled_dropoff|
+-------------------+-------------------+-------------------+-------------------+-------------+--------------+
|2001-01-01 00:06:49|2026-06-26 23:53:12|1970-01-20 10:16:32|2026-06-27 20:59:10|128202548    |128202548     |
+-------------------+-------------------+-------------------+-------------------+-------------+--------------+



In [18]:
df_taxi = df_taxi.filter(
    F.col("tpep_pickup_datetime") >= "2023-01-01"


)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 21, Finished, Available, Finished, False)

In [19]:
df_taxi = df_taxi.filter(
    (F.col("tpep_pickup_datetime") < "2026-01-01") &
    (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
)



StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 22, Finished, Available, Finished, False)

In [20]:
df_taxi = df_taxi.withColumn(
    "duration_minutes",
    F.expr(
        "timestampdiff(SECOND, tpep_pickup_datetime, "
        "tpep_dropoff_datetime) / 60.0"
    )
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 23, Finished, Available, Finished, False)

In [21]:
display(
    df_taxi.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "duration_minutes",
        "trip_distance",
        "total_amount"
    )
    .orderBy(F.col("duration_minutes").desc())
    .limit(10)
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 495786aa-d9cb-4898-9314-a1165f8cc4a1)

In [22]:
df_taxi = df_taxi.filter((F.col("duration_minutes") >= 0) &(F.col("duration_minutes") <= 1440))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 25, Finished, Available, Finished, False)

In [23]:
df_taxi = df_taxi.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

df_taxi = df_taxi.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 26, Finished, Available, Finished, False)

In [24]:
df_taxi.agg(
    F.count(F.when(F.col("trip_distance") == 0, 1)).alias("zero_distance"),
    F.count(F.when(F.col("trip_distance") < 0, 1)).alias("negative_distance"),
    F.count(F.when(F.col("total_amount") == 0, 1)).alias("zero_amount"),
    F.count(F.when(F.col("total_amount") < 0, 1)).alias("negative_amount")
    
).show()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 27, Finished, Available, Finished, False)

+-------------+-----------------+-----------+---------------+
|zero_distance|negative_distance|zero_amount|negative_amount|
+-------------+-----------------+-----------+---------------+
|      2952380|                0|      17795|        1959860|
+-------------+-----------------+-----------+---------------+



In [25]:
display(
    df_taxi.filter(F.col("total_amount") < 0)
    .select(
        "payment_type",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "trip_distance",
        "duration_minutes"
    )
    .limit(10)
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, caee736c-1ec2-44b9-b78a-d73f742e1e29)

In [26]:
display(
    df_taxi.select(
        "trip_distance",
        "duration_minutes",
        "total_amount",
        "ratecodeid"
    )
    .orderBy(F.col("trip_distance").desc())
    .limit(10)
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 718595b1-a91c-4164-a96f-0b927acafcbe)

In [27]:
print(df_taxi.filter(F.col("trip_distance") > 1000).count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 31, Finished, Available, Finished, False)

3973


In [28]:
df_taxi = df_taxi.withColumn(
    "trip_distance",
    F.when(
        F.col("trip_distance") > 1000,
        F.lit(None).cast("double")
    ).otherwise(F.col("trip_distance"))
)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 32, Finished, Available, Finished, False)

In [31]:
display(df_taxi.groupBy("passenger_count").count().orderBy("passenger_count"))

display(df_taxi.groupBy("ratecodeid").count().orderBy("ratecodeid"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a8470d92-f431-4a78-931d-c01c639d8558)

SynapseWidget(Synapse.DataFrame, 581f25c4-e67b-4596-982c-9f33a935b5dc)

In [30]:
df_taxi = df_taxi.withColumn("passenger_count", F.col("passenger_count").cast("int"))
df_taxi = df_taxi.withColumn("ratecodeid", F.col("ratecodeid").cast("int"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 34, Finished, Available, Finished, False)

In [32]:
zones = spark.table("lh_nyc_data.dbo.silver_taxi_zones")

zone_ids = zones.select(F.col("LocationID").alias("zone_id"))

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 36, Finished, Available, Finished, False)

In [34]:
zones = spark.table("lh_nyc_data.dbo.silver_taxi_zones")

pickup_ids = df_taxi.select("pulocationid").distinct()
dropoff_ids = df_taxi.select("dolocationid").distinct()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 38, Finished, Available, Finished, False)

In [38]:
print(df_taxi.filter(
        (F.abs(F.col("total_amount")) > 10000) |
        (F.abs(F.col("fare_amount")) > 10000)
    ).count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 42, Finished, Available, Finished, False)

14


In [39]:
df_taxi = df_taxi.filter(
    (F.abs(F.col("total_amount")) <= 10000) &
    (F.abs(F.col("fare_amount")) <= 10000))
    

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 43, Finished, Available, Finished, False)

In [40]:
rows = df_taxi.count()
unique_rows = df_taxi.dropDuplicates().count()

print("all rows", rows)
print("repeated", rows - unique_rows)

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 44, Finished, Available, Finished, False)

all rows 128195308
repeated 7


In [41]:
df_taxi = df_taxi.dropDuplicates()

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 45, Finished, Available, Finished, False)

In [42]:
df_taxi.write.format("delta").mode("overwrite").saveAsTable("lh_nyc_data.dbo.silver_taxi_trips")

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 46, Finished, Available, Finished, False)

In [43]:
saved_taxi = spark.table("lh_nyc_data.dbo.silver_taxi_trips")
print(saved_taxi.count())

StatementMeta(, 03a765b9-ccb4-496d-8d06-9bf7b1531847, 47, Finished, Available, Finished, False)

128195301
